### Explanation of analysis:
#### Design used: sex, age, bmi, beta cell proportion (non-beta cell celltypes), disease
#### Filters: min 2 samples per condition must express gene to test, min 20 cells for cell types except late t1d beta cells
#### multiome data and sample MM_426 (npod id - 6278) excluded from analysis 

In [2]:
suppressPackageStartupMessages(library(DESeq2))
suppressPackageStartupMessages(library(ggplot2))

In [ ]:
##### READ IN METADATA

wd_meta='/downstream_analysis/'

meta.data =read.table(paste0(wd_meta,'nPOD_snRNA_metadata.txt'),sep = '\t')
meta.data$status <- NULL
meta.data$status[meta.data$Condition_subtype =='T1D_early'] <-'3.T1D_early'
meta.data$status[meta.data$Condition_subtype =='T1D_late' ] <-'4.T1D_late'
meta.data$status[meta.data$Condition_subtype == 'Aab'] <-'2.Autoab+'
meta.data$status[meta.data$Condition_subtype == 'ND'] <-'1.Control'

####### SCALE ALL THE CONTINUOUS VARIABLES
meta.data$AgeScaled <- scale(as.numeric(meta.data$Age))
meta.data$BMIScaled <- scale(as.numeric(meta.data$BMI))

for(i in seq(18,35)){
    col_name <- colnames(meta.data)[i]
    new_name <- paste0(col_name,'_scaled')
    meta.data[new_name]<- as.double(scale(meta.data[i]))

}

for(i in seq(37,54)){
    col_name <- colnames(meta.data)[i]
    new_name <- paste0(col_name,'_scaled')
    meta.data[new_name]<- as.double(scale(meta.data[i]))

}

for(i in seq(55,72)){
    col_name <- colnames(meta.data)[i]
    new_name <- paste0(col_name,'_scaled')
    meta.data[new_name]<- as.double(scale(meta.data[i]))
}

for(i in seq(73,90)){
    col_name <- colnames(meta.data)[i]
    new_name <- paste0(col_name,'_scaled')
    meta.data[new_name]<- as.double(scale(meta.data[i]))

}

rownames(meta.data)<- meta.data$SeqID.RNA
meta.data$Immune_proportion <- rowSums(meta.data[,c('Bcells','Macrophage','Mast','Tcells')])/meta.data[,"Total"]
meta.data$Immune_proportion_scaled <- as.double(scale(meta.data$Immune_proportion))

head(meta.data)

## Donor must have at least 20 cells in tested cell type to be analyzed -- everything except late t1d beta cells

In [ ]:
# tpm or pseudobulk matrices directory
dir = '/data/PseudobulkMatrices/SoupX/min20cellMtx/'

# get list of files, remov"ing files with <200 cells or not enough data to test all conditions
files = list.files(dir,pattern=".txt")
files <-files[!files %in% c("Bcells_soupX_min20cells.txt",'LymphEndo_soupX_min20cells.txt')] #too few conditions or samples or cells to test
files
# cut off file suffices to get celltype names
cells = gsub("_soupX_min20cells.txt","", files)
cells 

In [ ]:
designs <- 1
#CT_use = CT_v2 ## Relative contamination 
outdir <- paste0('/downstream_analysis/RNA_deseq/')
dir.create(outdir)

for(design in designs) {
    ## create result matrix 
    sumres = matrix(nrow=length(cells), ncol = 5)
    rownames(sumres)= cells
    
    outdir <- paste0('/downstream_analysis/RNA_deseq/SoupX_min20Cells_min2SampleGeneFilt/')
    dir.create(outdir)

    for (FILE in files) {
        cell = cells[which(files == FILE)]
        raw_counts = read.table(paste0(dir, FILE), row.names=1, check.names=FALSE)

        removed_samples <- c('multi_6220','multi_6228','multi_6229','multi_6234','multi_6236','multi_6267',
                'multi_6362','multi_6375','6278')

        raw_counts <- raw_counts[,!colnames(raw_counts) %in% removed_samples]
        ### read in counts, pre filter -- remove samples with 0 counts for all genes for this cell type
        raw_counts      = raw_counts[, colSums(raw_counts != 0) > 0]
        meta_cell       = subset(meta.data, Sample.ID %in% colnames(raw_counts))
        meta_cell       = meta_cell[!duplicated(meta_cell$Sample.ID), ]

        rownames(meta_cell) <- meta_cell$Sample.ID
        raw_counts      = subset(raw_counts, select = as.character(meta_cell$Sample.ID))
        
        ### Create dfs of each conditions samples counts
        nd_raw_counts       <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'ND']))
        T1Dearly_raw_counts <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'T1D_early']))
        T1Dlate_raw_counts  <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'T1D_late']))
        Aab_raw_counts      <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'Aab']))
        
        ### remove any condition that doesn't have replicates from metadata
        if(ncol(nd_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(nd_raw_counts))
        } else {print("There's more than one sample in ND condition")}
        if(ncol(T1Dearly_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(T1Dearly_raw_counts))
        } else {print("There's more than one sample in T1D early condition")}
        if(ncol(T1Dlate_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(T1Dlate_raw_counts))
        } else {print("There's more than one sample in T1D late condition")}
        if(ncol(Aab_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(Aab_raw_counts))
        } else {print("There's more than one sample in Aab condition")}
        
        ### filter count matrix to reflect new metadata file
        raw_counts      = subset(raw_counts, select = as.character(meta_cell$Sample.ID))
        if(ncol(raw_counts)== 0 ) next # if no Samples left and go to next iteration
        if(! "ND" %in% meta_cell$Condition_subtype ) next # if no ND samples left and go to next iteration
        if(! "T1D_early" %in% meta_cell$Condition_subtype ) next # if no early T1D left and go to next iteration

        ### get a list for each condition of genes with >1 counts in at least 2 of the samples per condition
        n_ND   <- 2
        n_eT1D <- 2
        n_lT1D <- 2
        n_Aab  <- 2

        
        nd_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(nd_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(nd_raw_counts[i, ] >= 1) >= n_ND) {
            
            # add the row name to the result list
            nd_genes_to_keep <- c(nd_genes_to_keep, rownames(nd_raw_counts[i, ]))
          }
        }
        
        t1de_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(T1Dearly_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(T1Dearly_raw_counts[i, ] >= 1) >= n_eT1D) {
            
            # add the row name to the result list
            t1de_genes_to_keep <- c(t1de_genes_to_keep, rownames(T1Dearly_raw_counts[i, ]))
          }
        }
        
        t1dl_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(T1Dlate_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(T1Dlate_raw_counts[i, ] >= 1) >= n_lT1D) {
            
            # add the row name to the result list
            t1dl_genes_to_keep <- c(t1dl_genes_to_keep, rownames(T1Dlate_raw_counts[i, ]))
          }
        }
        
        aab_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(Aab_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(Aab_raw_counts[i, ] >= 1) >= n_Aab) {
            
            # add the row name to the result list
            aab_genes_to_keep <- c(aab_genes_to_keep, rownames(Aab_raw_counts[i, ]))
          }
        }
        
        

        
        ### If statements to perform multiple designs without a shit load of code
        if (design == 1){
            if (cell == 'Beta'){
                cellCount_scaled = paste0(cell,'_scaled')
                MTpercent_scaled = paste0(cell,'_meanPercentMT_scaled')
                MeanGene_scaled = paste0(cell,'_meanGene_scaled')
                f_string = as.formula(paste0("~ Sex + AgeScaled + BMIScaled  + status"))
                rr_beta = as.formula ("~ Sex + AgeScaled + BMIScaled ")
                my_design = f_string
                reduction = rr_beta
            }else{
                cellCount_scaled = paste0(cell,'_scaled')
                MTpercent_scaled = paste0(cell,'_meanPercentMT_scaled')
                MeanGene_scaled = paste0(cell,'_meanGene_scaled')

                f_string = paste0("~ Sex + AgeScaled + BMIScaled + Beta_proportion_scaled + status")
                r_string = paste0("~ Sex + AgeScaled + BMIScaled + Beta_proportion_scaled")
        
                ff = as.formula (f_string)
                rr = as.formula (r_string)
                my_design = ff
                reduction = rr 
            }               
        } else {
            print("Not Running Right")
        }
        ### Run DEseq    
        if ("4.T1D_late" %in% meta_cell$status){
                tests = c('2.Autoab+', '3.T1D_early', '4.T1D_late' )
                print(cell)
                print("all conditions found")
                print(my_design)
                print(reduction)
                ## light pre-filtering
                counts = raw_counts[rowSums(raw_counts) >= 10,]
                dds  <- DESeqDataSetFromMatrix(round(counts), colData = meta_cell, design = my_design)
                rld <- rlog(dds)
                print(plotPCA(rld, intgroup = "status")+ ggtitle(cell))
            ### LRT test
                dds <- estimateSizeFactors(dds)
                dds <- estimateDispersions(dds)
                dds <- nbinomLRT(dds, reduced=reduction)
                
                res    <- results(dds)
                res    = as.data.frame(res)
                res    = res[order(res$pvalue),]
            
                outfile = paste0( cell, ".deseq.LRT.tsv" )
                write.table(res[,-2],paste0(outdir, outfile) , sep="\t", quote=F) 
                sumres[cell,1]<-   sum(res$padj<0.1, na.rm=T) 
          
             
                ### Pairwise_test
               
                for(y in 1:3){
                    if ( y == 1) {
                    testingVar_keepGenes = aab_genes_to_keep
                    } else if ( y == 2) {
                    testingVar_keepGenes = t1de_genes_to_keep
                    } else {
                    testingVar_keepGenes = t1dl_genes_to_keep
                    }
            
                counts = raw_counts[rowSums(raw_counts) >= 10,]
                print(paste0("Number of rows prefiltering: ", nrow(counts)))

                counts = subset(counts, rownames(counts) %in% nd_genes_to_keep)
                print(paste0("Number of rows filtering ND genes: ", nrow(counts)))

                counts = subset(counts, rownames(counts) %in% testingVar_keepGenes)
                print(paste0("Number of rows filtering ND genes: ", nrow(counts)))

                dds  <- DESeqDataSetFromMatrix(round(counts), colData = meta_cell, design = my_design)
                dds <- estimateSizeFactors(dds)
                dds <- estimateDispersions(dds)
                dds2 <- dds                    
                dds2 <- nbinomWaldTest(dds2)

                res <- results(dds2, contrast=c("status",tests[y],"1.Control"),  independentFiltering=FALSE)
                res    = as.data.frame(res)
                res    = res[order(res$pvalue),]
        
                outfile = paste0( cell, ".deseq.", tests[y]  , ".tsv" )
                write.table(res,paste0(outdir, outfile) , sep="\t", quote=F) 
                sumres[cell,(y+1)]<-   sum(res$padj<0.1, na.rm=T) 
                
            }
            sumres[cell,5]<- ncol(raw_counts)   
            
        }else {
            tests = c('2.Autoab+', '3.T1D_early' )
            print(cell)
            print("no late T1D")
            print(my_design)
            print(reduction)
            ## light pre-filtering
            counts = raw_counts[rowSums(raw_counts) >= 10,]

            #normMatrix <- contam
    
    
            #my_design = ff
   
            dds  <- DESeqDataSetFromMatrix(round(counts), colData = meta_cell, design = my_design)
            rld <- rlog(dds)
            print(plotPCA(rld, intgroup = "status")+ ggtitle(cell))
            ### LRT test
            dds <- estimateSizeFactors(dds)
            dds <- estimateDispersions(dds)
            dds <- nbinomLRT(dds, reduced=reduction)
        
            res    <- results(dds)
            res    = as.data.frame(res)
            res    = res[order(res$pvalue),]
            
            outfile = paste0( cell, ".deseq.LRT.tsv" )
            write.table(res[,-2],paste0(outdir, outfile) , sep="\t", quote=F) 
            sumres[cell,1]<-   sum(res$padj<0.1, na.rm=T) 
          
             
            ### Pairwise_test
               
            dds2 <- nbinomWaldTest(dds2)
            for(y in 1:2){
                if ( y == 1) {
                testingVar_keepGenes = aab_genes_to_keep
                } else {
                testingVar_keepGenes = t1de_genes_to_keep
                } 
            
                counts = raw_counts[rowSums(raw_counts) >= 10,]
                print(paste0("Number of rows prefiltering: ", nrow(counts)))

                counts = subset(counts, rownames(counts) %in% nd_genes_to_keep)
                print(paste0("Number of rows filtering ND genes: ", nrow(counts)))

                counts = subset(counts, rownames(counts) %in% testingVar_keepGenes)
                print(paste0("Number of rows filtering ND genes: ", nrow(counts)))

                dds  <- DESeqDataSetFromMatrix(round(counts), colData = meta_cell, design = my_design)
                dds <- estimateSizeFactors(dds)
                dds <- estimateDispersions(dds)
                dds2 <- dds                    
                dds2 <- nbinomWaldTest(dds2)

                res <- results(dds2, contrast=c("status",tests[y],"1.Control"),  independentFiltering=FALSE)
                res    = as.data.frame(res)
                res    = res[order(res$pvalue),]
        
                outfile = paste0( cell, ".deseq.", tests[y]  , ".tsv" )
                write.table(res,paste0(outdir, outfile) , sep="\t", quote=F) 
                sumres[cell,(y+1)]<-   sum(res$padj<0.1, na.rm=T) 
                
            }
            sumres[cell,5]<- ncol(raw_counts)
        }   
        
        
        
          
        tests = c('2.Autoab+', '3.T1D_early', '4.T1D_late' )
        colnames(sumres) = c( "LRT",tests, "nsamples")
        write.csv(sumres, paste0(outdir, "Summary_table1.csv"))
    
         
        print(sumres)

        
    }
}

## late t1d beta cell analysis -- using all donors regardless of number of cells

In [ ]:
# tpm or pseudobulk matrices directory
dir = '/data/PseudobulkMatrices/SoupX/AllCellMtx/'

files ='Beta_soupX_allCells.txt'
cells = gsub("_soupX_allCells.txt","", files)
cells  

In [ ]:
designs <- 1
#CT_use = CT_v2 ## Relative contamination 
outdir <- paste0('/downstream_analysis/RNA_deseq/')
dir.create(outdir)

for(design in designs) {
    ## create result matrix 
    sumres = matrix(nrow=length(cells), ncol = 3)
    rownames(sumres)= cells
    
    outdir <- paste0('/downstream_analysis/RNA_deseq/SoupX_min20Cells_min2SampleGeneFilt/LateT1D/')
    dir.create(outdir)

    for (FILE in files) {
        cell = cells[which(files == FILE)]
        raw_counts = read.table(paste0(dir, FILE), row.names=1, check.names=FALSE)

        removed_samples <- c('multi_6220','multi_6228','multi_6229','multi_6234','multi_6236','multi_6267',
                'multi_6362','multi_6375','6278') # remove multiome samples

        raw_counts <- raw_counts[,!colnames(raw_counts) %in% removed_samples]
        ### read in counts, pre filter -- remove samples with 0 counts for all genes for this cell type
        raw_counts      = raw_counts[, colSums(raw_counts != 0) > 0]
        meta_cell       = subset(meta.data, Sample.ID %in% colnames(raw_counts))
        meta_cell       = meta_cell[!duplicated(meta_cell$Sample.ID), ]

        rownames(meta_cell) <- meta_cell$Sample.ID
        raw_counts      = subset(raw_counts, select = as.character(meta_cell$Sample.ID))
        
        ### Create dfs of each conditions samples counts
        nd_raw_counts       <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'ND']))
        T1Dearly_raw_counts <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'T1D_early']))
        T1Dlate_raw_counts  <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'T1D_late']))
        Aab_raw_counts      <- subset(raw_counts, select = as.character(meta_cell$Sample.ID[meta_cell$Condition_subtype == 'Aab']))
        
        ### remove any condition that doesn't have replicates from metadata
        if(ncol(nd_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(nd_raw_counts))
        } else {print("There's more than one sample in ND condition")}
        if(ncol(T1Dearly_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(T1Dearly_raw_counts))
        } else {print("There's more than one sample in T1D early condition")}
        if(ncol(T1Dlate_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(T1Dlate_raw_counts))
        } else {print("There's more than one sample in T1D late condition")}
        if(ncol(Aab_raw_counts) <= 1) {
            print("There's no replicates")
            meta_cell <- subset(meta_cell, ! rownames(meta_cell) %in% colnames(Aab_raw_counts))
        } else {print("There's more than one sample in Aab condition")}
        
        ### filter count matrix to reflect new metadata file
        raw_counts      = subset(raw_counts, select = as.character(meta_cell$Sample.ID))
        if(ncol(raw_counts)== 0 ) next # if no sig pathways and go to next iteration
        if(! "ND" %in% meta_cell$Condition_subtype ) next # if raw_counts is empty and go to next iteration
        if(! "T1D_early" %in% meta_cell$Condition_subtype ) next # if raw_counts is empty and go to next iteration

        ### get a list for each condition of genes with >5 counts in at least 50% of samples per condition
        n_ND   <- 2
        n_eT1D <- 2
        n_lT1D <- 2
        n_Aab  <- 2

        
        nd_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(nd_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(nd_raw_counts[i, ] >= 1) >= n_ND) {
            
            # add the row name to the result list
            nd_genes_to_keep <- c(nd_genes_to_keep, rownames(nd_raw_counts[i, ]))
          }
        }
        
        t1de_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(T1Dearly_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(T1Dearly_raw_counts[i, ] >= 1) >= n_eT1D) {
            
            # add the row name to the result list
            t1de_genes_to_keep <- c(t1de_genes_to_keep, rownames(T1Dearly_raw_counts[i, ]))
          }
        }
        
        t1dl_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(T1Dlate_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(T1Dlate_raw_counts[i, ] >= 1) >= n_lT1D) {
            
            # add the row name to the result list
            t1dl_genes_to_keep <- c(t1dl_genes_to_keep, rownames(T1Dlate_raw_counts[i, ]))
          }
        }
        
        aab_genes_to_keep <- c()
        # loop through each row in the data frame
        for (i in 1:nrow(Aab_raw_counts)) {
          
          # check if there are at least n_ND values greater than 5 in the current row
          if (sum(Aab_raw_counts[i, ] >= 1) >= n_Aab) {
            
            # add the row name to the result list
            aab_genes_to_keep <- c(aab_genes_to_keep, rownames(Aab_raw_counts[i, ]))
          }
        }
        
        

        
        ### If statements to perform multiple designs without a shit load of code
        if (design == 1){
            if (cell == 'Beta'){
                cellCount_scaled = paste0(cell,'_scaled')
                MTpercent_scaled = paste0(cell,'_meanPercentMT_scaled')
                MeanGene_scaled = paste0(cell,'_meanGene_scaled')
                f_string = as.formula(paste0("~ Sex + AgeScaled + BMIScaled  + status"))
                rr_beta = as.formula ("~ Sex + AgeScaled + BMIScaled ")
                my_design = f_string
                reduction = rr_beta
            }else{
                cellCount_scaled = paste0(cell,'_scaled')
                MTpercent_scaled = paste0(cell,'_meanPercentMT_scaled')
                MeanGene_scaled = paste0(cell,'_meanGene_scaled')

                f_string = paste0("~ Sex + AgeScaled + BMIScaled + Beta_proportion_scaled + status")
                r_string = paste0("~ Sex + AgeScaled + BMIScaled + Beta_proportion_scaled")
        
                ff = as.formula (f_string)
                rr = as.formula (r_string)
                my_design = ff
                reduction = rr 
            }               
        } else {
            print("Not Running Right")
        }
        ### Run DEseq    

                tests = c('4.T1D_late' )
                print(cell)
                print("all conditions found")
                print(my_design)
                print(reduction)
                ## light pre-filtering
                counts = raw_counts[rowSums(raw_counts) >= 10,]

                #normMatrix <- contam
    
    
                #my_design = ff
   
                dds  <- DESeqDataSetFromMatrix(round(counts), colData = meta_cell, design = my_design)
                rld <- rlog(dds)
                print(plotPCA(rld, intgroup = "status")+ ggtitle(cell))
            ### LRT test
                dds <- estimateSizeFactors(dds)
                dds <- estimateDispersions(dds)
                dds <- nbinomLRT(dds, reduced=reduction)
                
                res    <- results(dds)
                res    = as.data.frame(res)
                res    = res[order(res$pvalue),]
            
                outfile = paste0( cell, ".deseq.LRT.tsv" )
                write.table(res[,-2],paste0(outdir, outfile) , sep="\t", quote=F) 
                sumres[cell,1]<-   sum(res$padj<0.1, na.rm=T) 
          
             
                ### Pairwise_test
               
               y=1
                testingVar_keepGenes = t1dl_genes_to_keep
                    
            
                counts = raw_counts[rowSums(raw_counts) >= 10,]
                print(paste0("Number of rows prefiltering: ", nrow(counts)))

                counts = subset(counts, rownames(counts) %in% nd_genes_to_keep)
                print(paste0("Number of rows filtering ND genes: ", nrow(counts)))

                counts = subset(counts, rownames(counts) %in% testingVar_keepGenes)
                print(paste0("Number of rows filtering ND genes: ", nrow(counts)))

                dds  <- DESeqDataSetFromMatrix(round(counts), colData = meta_cell, design = my_design)
                dds <- estimateSizeFactors(dds)
                dds <- estimateDispersions(dds)
                dds2 <- dds                    
                dds2 <- nbinomWaldTest(dds2)

                res <- results(dds2, contrast=c("status",tests[y],"1.Control"),  independentFiltering=FALSE)
                res    = as.data.frame(res)
                res    = res[order(res$pvalue),]
        
                outfile = paste0( cell, ".deseq.", tests[y]  , ".tsv" )
                write.table(res,paste0(outdir, outfile) , sep="\t", quote=F) 
                sumres[cell,(y+1)]<-   sum(res$padj<0.1, na.rm=T) 
                
            
            sumres[cell,3]<- ncol(raw_counts)   
            
       
        
        
          
        tests = c('4.T1D_late' )
        colnames(sumres) = c( "LRT",tests, "nsamples")
        write.csv(sumres, paste0(outdir, "Summary_table1.csv"))
    
         
        print(sumres)

        
    }
}


In [3]:
sessionInfo()

R version 4.1.1 (2021-08-10)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 20.04.2 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.9.0
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.9.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] ggplot2_3.5.0               DESeq2_1.34.0              
 [3] SummarizedExperiment_1.24.0 Biobase_2.54.0             
 [5] MatrixGenerics_1.6.0        matrixStats_1.2.0          
 [7] GenomicRanges_1.46.1        GenomeInfoDb_1.39.8        
 [9] IRang